# CE49X: Introduction to Computational Thinking and Data Science for Civil Engineers
## Week 10: Neural Networks

**Instructor:** Dr. Eyuphan Koc
**Department of Civil Engineering, Bogazici University**
**Semester:** Spring 2026

Based on *Hands-On Machine Learning with Scikit-Learn, Keras & TensorFlow* (3rd ed.) by Aurélien Géron,
Chapter 10: Introduction to Artificial Neural Networks with Keras

---

In [ ]:
# Standard imports — same stack you have used since Week 3.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings

# Reproducibility & figure defaults (consistent with Weeks 6–9).
np.random.seed(49)
plt.rcParams['figure.dpi'] = 100
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# Models & utilities.
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, validation_curve
from sklearn.metrics import (accuracy_score, classification_report,
                             ConfusionMatrixDisplay)
from sklearn.datasets import make_moons, make_regression

## Quick Recap: Where Are We?

Over the last four weeks you met four families of supervised models:

- **Logistic / linear regression** (Week 6) — straight lines and S-curves; calibrated probabilities; the simplest baseline.
- **Naive Bayes** (Week 7) — a *generative* model that asks *what does each class look like?* and fits a probability distribution per class.
- **Support Vector Machines** (Week 8) — a *discriminative* model that asks *where is the widest gap between classes?*, optionally bent into curves with **kernels**.
- **Decision Trees & Random Forests** (Week 9) — sequential yes/no questions; native handling of categoricals; native feature importances.

This week we meet the fifth family: **Neural Networks**. Where SVM kernels were *hand-picked* feature transforms (RBF, polynomial, …), neural networks **learn their feature transforms from data**. That single shift is the conceptual seed for everything called "deep learning" in 2026.

## Table of Contents

1. [Introduction: From Hand-Picked to Learned Features](#1.-Introduction:-From-Hand-Picked-to-Learned-Features)
2. [The Perceptron: One Neuron at a Time](#2.-The-Perceptron:-One-Neuron-at-a-Time)
3. [Why One Neuron Isn't Enough: Hidden Layers](#3.-Why-One-Neuron-Isn't-Enough:-Hidden-Layers)
4. [How the Network Learns: Loss, Gradient Descent & Backprop](#4.-How-the-Network-Learns:-Loss,-Gradient-Descent-&-Backprop)
5. [Application: MLP vs Random Forest on Nepal Earthquake Damage](#5.-Application:-MLP-vs-Random-Forest-on-Nepal-Earthquake-Damage)
6. [Regression, Comparison, Takeaways & Practice](#6.-Regression,-Comparison,-Takeaways-&-Practice)

---

## 1. Introduction: From Hand-Picked to Learned Features

### Why neural networks now?

For most of this course, every "feature" you fed a model was something *you* designed:
the water-cement ratio (Week 1), the wave-energy efficiency (Week 2), the foundation
type (Week 9). When you used SVM with an RBF kernel (Week 8), you were *implicitly*
projecting your data into a much richer space — but you still had to **pick the kernel
by hand**.

A neural network flips that responsibility. It takes raw inputs and **learns its own
internal feature transforms** layer by layer, end-to-end with the prediction objective.
For tabular civil-engineering data, this is rarely a game-changer (we'll see why on
Nepal). But for **images** (drone photos of post-quake buildings), **time series**
(accelerometer signals from structural-health-monitoring systems), and **text**
(inspector reports), learned representations are the entire reason deep learning
became dominant.

> **Key Insight: Kernels are *hand-picked* feature transforms; hidden layers are
> *learned* feature transforms.** This single sentence is the conceptual bridge from
> Week 8 to today.

### The philosophy: why "neural" networks?

Before any math, it helps to know *why* this family of models is named after the
brain. The metaphor is older than the math, and it still shapes how people talk
about the field in 2026.

In the 1940s and 50s — long before computers could fit on a desk — McCulloch &
Pitts (1943) and Rosenblatt (1958) asked a strange engineering question:

> *Can we build something useful by mimicking, even crudely, how the brain seems
> to work?*

The brain has roughly 86 billion **neurons**. Each neuron receives signals from
thousands of others, sums them up, and **fires** (sends an electrical spike) only
when the total signal crosses some threshold. Cognition — recognising your
grandmother, parsing a sentence, deciding to brake — somehow emerges from the
*patterns of firing* across this enormous network. No single neuron "is" your
grandmother; the concept lives in a distributed pattern of co-activations.

That observation made for an audacious bet: **maybe intelligence isn't a single
clever algorithm, but an *emergent property* of many simple units interacting**.
The artificial neural network is the engineer's stripped-down version of that
bet — a few mathematical "neurons" wired together, trained until useful patterns
of firing emerge.

> **Honest disclaimer.** Modern neural networks are *not* models of biological
> brains. Real neurons are vastly more complex (timing, neurotransmitters,
> plasticity rules), and a 2026 large language model bears about as much
> resemblance to your cortex as a paper airplane does to a falcon. The "neural"
> name is a historical accident that stuck. The math is just optimisation.

What *did* survive from the metaphor — and what genuinely matters — are two
ideas:

1. **Distributed representation.** No single weight or neuron carries "the"
   meaning. Concepts are encoded as *patterns across many neurons*. This is
   why neural networks tend to fail gracefully (a few broken neurons barely
   hurt) and why they generalise: similar inputs activate similar patterns.
2. **Layers of abstraction.** Stack neurons in layers, and (with training) the
   early layers tend to detect low-level patterns and later layers combine
   them into higher-level concepts. In a vision model trained on photos:
   *layer 1 → edges; layer 2 → textures; layer 3 → object parts; layer 4 →
   whole objects*. This mirrors what neuroscientists see in the visual
   cortex — convergent evolution between math and biology.

> **Key Insight: Power comes from many simple units, layered.** A single neuron
> does almost nothing useful. The interesting behaviour — the *thing that learns
> features for the task at hand* — emerges only when you stack many of them and
> let an optimiser tune their weights together.

Today's lecture is this idea in miniature. We start with one neuron (boring),
add a hidden layer (suddenly powerful), and watch a network *invent its own
features* on a curved 2-D problem. Every concept that powers ChatGPT or a
self-driving car's vision system is already present in what we'll do here —
just with more layers and more data.

### Honest framing: why we use sklearn (not Keras) today

`sklearn.neural_network.MLPClassifier` is a **dense, CPU-only, multi-layer perceptron**.
It is deliberately small. It does **not** support:

- Convolutional layers (CNNs for images).
- Recurrent or attention layers (RNNs / Transformers for sequences).
- GPU acceleration.
- Custom architectures.

We use it because:

1. **Zero new dependencies** — the same stack you already have.
2. **Focuses attention on the ideas** — perceptron, hidden layers, activations, loss,
   gradient descent, backprop — without any new framework syntax to learn.
3. **Honest scope** — when you finish today, you will know *exactly* when you would
   reach for Keras / PyTorch and what they buy you. We map this in §6.

This is a feature, not a bug. The concepts you learn here transfer directly to
deep-learning frameworks; only the engineering (GPUs, custom layers, scale) changes.

## 2. The Perceptron: One Neuron at a Time

### A single neuron is a familiar object

Strip a neural network down to its smallest unit and you find this: a **neuron** takes
a vector of inputs $x_1, x_2, \ldots, x_n$, multiplies each by a learned **weight**
$w_i$, sums them with a learned **bias** $b$, and passes the result through a
non-linear **activation function** $\sigma$:

$$\hat{y} = \sigma\!\left(w_1 x_1 + w_2 x_2 + \cdots + w_n x_n + b\right)
        = \sigma\!\left(\mathbf{w}^\top \mathbf{x} + b\right)$$

In words: *weight each input by how much it matters, add them up with a baseline lean,
then squash.*

> **Definition: Perceptron.** A single neuron with a chosen activation function.
> A perceptron with a sigmoid activation is *exactly* logistic regression — you
> already trained one in Week 6.

### A civil-engineering analogy

> **Example: Civil Engineering.** Imagine a junior structural inspector triaging
> buildings after a moderate earthquake. They weight a few features (foundation
> condition, age, height, material), add them up with a baseline tolerance, and
> threshold:
>
> $$\text{score} = 0.6 \cdot \text{foundation\_cracked} + 0.3 \cdot \text{age\_over\_50} + 0.2 \cdot \text{heavy\_roof} - 0.1$$
>
> $$\text{action} = \begin{cases} \text{flag for inspection} & \text{if } \sigma(\text{score}) > 0.5 \\ \text{ok} & \text{otherwise} \end{cases}$$
>
> A perceptron is *exactly* this rule, except the computer will learn the four
> numbers $(0.6, 0.3, 0.2, -0.1)$ from data instead of you guessing them.

### The activation zoo

Why do we need a non-linear $\sigma$? Because **stacking linear functions just gives
another linear function**. Without a non-linearity, a 50-layer network is just a
fancy linear regression. The non-linearity is what lets layers do useful work
on top of each other.

Three activations dominate:

- **ReLU** ($\max(0, z)$) — modern default. Cheap, doesn't saturate for positive
  inputs, and works extremely well in practice. Used inside hidden layers.
- **Sigmoid** ($1/(1 + e^{-z})$) — squashes to $(0, 1)$. Used at the output of a
  *binary* classifier to produce a probability.
- **Tanh** ($\tanh(z)$) — squashes to $(-1, 1)$. A centred squasher; sometimes used
  in hidden layers, mostly historical.

In [ ]:
# Plot the three activations on a single axes — Week-9 colour palette.
z = np.linspace(-5, 5, 400)

relu    = np.maximum(0, z)
sigmoid = 1 / (1 + np.exp(-z))
tanh    = np.tanh(z)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(z, relu,    color='steelblue', linewidth=2.5, label='ReLU  $\\max(0, z)$')
ax.plot(z, sigmoid, color='indianred', linewidth=2.5, label='Sigmoid  $1/(1+e^{-z})$')
ax.plot(z, tanh,    color='gray',      linewidth=2.5, linestyle='--', label='Tanh  $\\tanh(z)$')
ax.axhline(0, color='black', linewidth=0.5)
ax.axvline(0, color='black', linewidth=0.5)
ax.set_xlabel('input  $z = \\mathbf{w}^\\top \\mathbf{x} + b$', fontsize=12)
ax.set_ylabel('activation $\\sigma(z)$', fontsize=12)
ax.set_title('Three activation functions', fontsize=13)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11, loc='upper left')
plt.tight_layout()
plt.show()

### `[LIVE]` A neuron by hand

Let's make this completely concrete. Suppose a neuron has three inputs (say, three
features extracted from a damaged beam: crack length, deflection, and load applied),
three weights, a bias, and a sigmoid activation. We pick the weights by hand and
compute the output for one example:

In [ ]:
# Three inputs (e.g. crack length [mm], deflection [mm], applied load [kN]).
x = np.array([2.0, 1.5, 0.8])

# Hand-picked weights and bias (in a real model, gradient descent would learn these).
w = np.array([0.6, -0.3, 0.2])
b = -0.1

# Step 1: weighted sum.
z = w @ x + b
print(f"Weighted sum  z = w · x + b = {z:.4f}")

# Step 2: apply activations.
print(f"ReLU(z)    = {max(0.0, z):.4f}")
print(f"Sigmoid(z) = {1 / (1 + np.exp(-z)):.4f}   (interpret as P(damage))")

> **Key Insight: A single sigmoid neuron is logistic regression.** No more, no less.
> The mystique around neural networks comes entirely from *stacking* neurons into
> layers — which is what we'll do next.

### What does it mean for a neuron to "fire"?

The vocabulary we just borrowed from biology — *firing* — has a precise meaning
in the artificial version. A neuron's output, $\sigma(\mathbf{w}^\top
\mathbf{x} + b)$, is its **firing rate** for the current input. The activation
function decides exactly how the neuron fires:

- **ReLU neuron** (modern default). Fires only when $z = \mathbf{w}^\top
  \mathbf{x} + b > 0$. If $z \le 0$, the output is *exactly zero* — the neuron
  is **silent**, contributing nothing to anything downstream. This is why ReLU
  networks are *sparse*: for any given input, only a subset of hidden neurons
  are actually firing; the rest are off.
- **Sigmoid neuron**. Fires "softly" — output is anywhere in $(0, 1)$. Read it
  as the neuron's *confidence* that the pattern it cares about is present.
- **Tanh neuron**. Fires positively or negatively in $(-1, 1)$ — useful when
  you want a neuron to push the next layer up *or* down.

### Each neuron is a tiny pattern detector

Once a neuron is trained, its weights $\mathbf{w}$ encode **a specific pattern
in the input it cares about**. The neuron fires strongly when the input matches
that pattern (the weighted sum is large and positive) and stays quiet otherwise.

In our hand-calculation above, the weight vector $\mathbf{w} = [0.6, -0.3,
0.2]$ defined a neuron that fires for examples with **large crack length**,
**small deflection**, and **moderate load** — that's its pattern. A different
neuron in the same layer, with different weights, will care about a different
pattern.

> **Definition: A neuron's firing pattern is its specialty.** The weight vector
> $\mathbf{w}$ *is* the description of which inputs make it fire. Training is
> the process of nudging every neuron's weights until each one specialises in a
> pattern useful for the task.

### How firing contributes to the network as a whole

The magic of layering is what happens when many specialised firings combine:

```
   raw inputs   →   hidden layer 1        →   hidden layer 2        →   output
                  (each neuron fires       (each neuron fires
                   for some low-level       for a useful combination
                   pattern of inputs)       of layer-1 firings)
```

For any given input, the network produces a **firing pattern** — a vector
recording *which* hidden neurons fired and *how strongly*. That firing pattern
is the network's internal **fingerprint** of the input. Two inputs that share a
fingerprint will be classified the same way, even if their raw features look
different — and this is exactly the property that lets the network generalise.

Concretely, on a Nepal building:

- Some hidden-layer-1 neuron might fire whenever a building is *short* and
  *made of stone*.
- Another might fire on *recently built* and *high foundation quality*.
- A hidden-layer-2 neuron might fire when "stone & short" fires *and* "low
  foundation quality" fires — encoding a higher-level concept ("vulnerable
  rural construction").
- The output neuron looks at the layer-2 firing pattern and produces the final
  damage-grade prediction.

No engineer wrote those concepts down. The network *discovered* them by
adjusting weights to lower the loss. This is what we mean by **learned
features** — and it is the entire point of building a network instead of doing
logistic regression on the raw inputs.

> **Key Insight: A neuron firing is a vote that "my pattern is present."**
> The network's prediction is what you get when you let many such votes compose
> through layers. Stop thinking of neurons as opaque dots in a diagram — each
> one is a small, specialised detector, and the network's intelligence lives in
> the *combinations of firings*, not in any single unit.

### `[QUICK]` Sanity check

Before we move on, two questions to answer in your head:

1. We use **sigmoid** at the *output* of a binary classifier (it produces a
   probability in $(0, 1)$). Which activation would you use at the output of a
   *regression* model — sigmoid, ReLU, tanh, or none? Why?
2. If you stack two ReLU neurons but remove the ReLU (i.e. set $\sigma$ = identity),
   what kind of model do you get? *(Hint: the answer is the reason we need
   non-linear activations at all.)*

## 3. Why One Neuron Isn't Enough: Hidden Layers

### The non-linearity wall

A single perceptron draws **one** straight line through the input space. Anything
that needs a curved boundary breaks it. This is the same wall we hit in Week 8 —
and the kernel trick was our first answer.

A neural network's answer is different: instead of *picking* a non-linear transform
(an RBF kernel), we **stack** neurons. The first layer (the **hidden layer**) takes
the raw inputs and computes several different weighted-sum-and-activation outputs —
each one is a "learned feature". The next layer combines those learned features
into a prediction. Picture:

```
        inputs           hidden layer            output
                       (feature inventor)        (decider)
        x₁  ─┐         ┌─ h₁ = σ(W₁₁x₁ + W₁₂x₂ + b₁) ─┐
              ├──────→  ├─ h₂ = σ(W₂₁x₁ + W₂₂x₂ + b₂) ─├──→  σ(v₁h₁ + v₂h₂ + v₃h₃ + c) = ŷ
        x₂  ─┘         └─ h₃ = σ(W₃₁x₁ + W₃₂x₂ + b₃) ─┘
```

Each $h_j$ is a *new feature* the network discovers. The output neuron then makes a
linear-then-squash decision on those new features. Because the hidden layer is
non-linear, the boundary in the *original* space can be arbitrarily curved.

### A two-moons demonstration

Let's watch this happen on `make_moons` — the same toy dataset Week 9 used to show
overfitting in trees. A linear classifier should fail; a tiny MLP should succeed.

In [ ]:
# Generate a non-linear 2D classification problem.
X_moons, y_moons = make_moons(n_samples=400, noise=0.20, random_state=49)

fig, ax = plt.subplots(figsize=(7, 5.5))
for cls, colour, label in [(0, 'steelblue', 'class 0'), (1, 'indianred', 'class 1')]:
    mask = y_moons == cls
    ax.scatter(X_moons[mask, 0], X_moons[mask, 1],
               c=colour, edgecolor='white', s=40, alpha=0.85, label=label)
ax.set_xlabel('$x_1$', fontsize=12)
ax.set_ylabel('$x_2$', fontsize=12)
ax.set_title('make_moons (noise=0.20)', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# A reusable decision-region plotter (adapted from Week 9's tree visualiser).
def visualize_decision_regions(model, X, y, ax, title=''):
    h = 0.02
    x_min, x_max = X[:, 0].min() - 0.4, X[:, 0].max() + 0.4
    y_min, y_max = X[:, 1].min() - 0.4, X[:, 1].max() + 0.4
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25,
                colors=['steelblue', 'indianred'])
    for cls, colour in [(0, 'steelblue'), (1, 'indianred')]:
        mask = y == cls
        ax.scatter(X[mask, 0], X[mask, 1], c=colour,
                   edgecolor='white', s=30)
    ax.set_title(title, fontsize=12)
    ax.set_xticks([]); ax.set_yticks([])

# Single perceptron baseline = logistic regression.
linear_baseline = LogisticRegression().fit(X_moons, y_moons)

# A small MLP with one hidden layer of 8 ReLU neurons.
mlp_small = MLPClassifier(hidden_layer_sizes=(8,), activation='relu',
                          max_iter=2000, random_state=49).fit(X_moons, y_moons)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
visualize_decision_regions(linear_baseline, X_moons, y_moons, axes[0],
                           f'Linear baseline (1 perceptron)\n'
                           f'accuracy = {linear_baseline.score(X_moons, y_moons):.2f}')
visualize_decision_regions(mlp_small,       X_moons, y_moons, axes[1],
                           f'MLP, 1 hidden layer of 8 ReLU\n'
                           f'accuracy = {mlp_small.score(X_moons, y_moons):.2f}')
plt.tight_layout()
plt.show()

The linear model slices straight through the moons; the small MLP carves a curved
boundary that follows them. **One hidden layer was enough** — the network invented
features that make the original problem linearly separable in its internal
representation.

### How wide should a hidden layer be?

Width controls expressive power, with the same bias-variance trade-off you saw with
tree depth in Week 9. Too few neurons → underfit. Too many → overfit.

In [ ]:
# Sweep hidden-layer width on moons (mirrors Week 9's depth-1/3/7/15 grid for trees).
widths = [(2,), (8,), (64,), (256,)]

fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for ax, hl in zip(axes.flat, widths):
    m = MLPClassifier(hidden_layer_sizes=hl, activation='relu',
                      max_iter=4000, random_state=49).fit(X_moons, y_moons)
    visualize_decision_regions(m, X_moons, y_moons, ax,
                               f'hidden_layer_sizes={hl}, accuracy={m.score(X_moons, y_moons):.2f}')
plt.tight_layout()
plt.show()

You can almost see the boundary go from rigid (2 neurons) to clean (8) to wiggly
and overfit (256). The lesson is identical to Week 9: **expressive capacity is a
dial, and you tune it just like tree depth or SVM's `C`**.

### `[TOGETHER]` Forward pass by hand

Let's compute one full forward pass through a 2-input, 2-hidden, 1-output network
with concrete numbers. This is the neural-network analog of Week 9's "Hand-Built
Split on a Tiny Beam Dataset".

In [ ]:
# Architecture: 2 inputs → 2 hidden (ReLU) → 1 output (sigmoid).
# Hand-picked weights and biases.
W1 = np.array([[ 0.5, -0.8],
               [ 0.3,  0.9]])      # shape (2 hidden, 2 inputs)
b1 = np.array([-0.1,  0.2])        # shape (2 hidden,)

W2 = np.array([[1.2, -0.7]])       # shape (1 output, 2 hidden)
b2 = np.array([0.05])              # shape (1 output,)

# An input example.
x = np.array([0.5, -0.2])
print(f"input x = {x}")

# Hidden layer: pre-activation z, then ReLU.
z1 = W1 @ x + b1
h  = np.maximum(0.0, z1)
print(f"  hidden pre-activation z1 = {z1}")
print(f"  hidden activation     h  = ReLU(z1) = {h}")

# Output layer: pre-activation z2, then sigmoid.
z2 = W2 @ h + b2
y_hat = 1 / (1 + np.exp(-z2))
print(f"  output pre-activation z2 = {z2}")
print(f"  output ŷ = sigmoid(z2)  = {y_hat}   (interpreted as P(class=1))")

### The same thing, in compact notation

That hand calculation, written compactly:

$$\mathbf{h} = \sigma\!\left(W^{(1)} \mathbf{x} + \mathbf{b}^{(1)}\right),
\qquad
\hat{y} = \sigma\!\left(W^{(2)} \mathbf{h} + b^{(2)}\right)$$

A network with $L$ hidden layers is just this pattern, applied $L+1$ times.

### A whisper of universal approximation

A famous result (Cybenko 1989, Hornik 1991) says that **a feedforward network with a
single hidden layer of enough neurons can approximate any continuous function on a
bounded domain**, to any accuracy you like. We will not prove it — but the practical
consequence is what matters:

- *Width* and *depth* control expressive power.
- The real question is never "can the network represent my function?" — it can.
  The question is **"do I have enough data to fit it without overfitting?"**

> **Key Insight: Hidden layers are *learned* feature transforms.** The kernels you
> hand-picked in Week 8 were a fixed non-linear transform; here, the network learns
> a transform that is tuned to your specific dataset. **This is the conceptual core
> of every deep-learning model.**

## 4. How the Network Learns: Loss, Gradient Descent & Backprop

We have a network. How does it find good weights? **Three ideas**, all of which
you have already met in different costumes:

1. **Loss function** — a single number that measures how wrong the network is.
2. **Gradient descent** — adjust weights in the direction that lowers the loss.
3. **Backpropagation** — efficiently compute *how much each weight contributed*
   to the loss, using the chain rule.

> **Walking-through analogy.** Think of training like *renovating a house with a
> very strict inspector*:
>
> 1. **Loss** = the inspector's verdict on a single page — *"how bad is it,
>    right now?"* One number, smaller is better.
> 2. **Gradient descent** = picking up your tools and *moving in the direction
>    that fixes the most damage per minute*. You don't try to fix everything at
>    once; you take a small step, ask the inspector again, take another step.
> 3. **Backprop** = the inspector's *itemised report*: which screw, beam, or
>    paint job is responsible for which fraction of the bad verdict, all the way
>    back from the visible damage to the original mistake. Without that report
>    you wouldn't know *what to fix first*.
>
> Lose any one of those three pieces and training falls apart. Together, they
> are the entire engine of every neural network ever trained — from the
> 1980s perceptron to the trillion-parameter language model.

We'll walk through them in that order, building intuition first and showing the
math just enough that you can debug a model that won't train. By the end of
this section you will be able to look at any `loss_curve_` and say *"that's
healthy"*, *"that's too aggressive"*, or *"that's not learning at all."*


### The loss function: a single number that says "how wrong?"

The first thing you need is a *score* the model can try to lower. For binary
classification we use **cross-entropy** (a.k.a. log-loss). The formula will
appear in a moment — but the verbal intuition is what matters:

> **Cross-entropy = the penalty for being confidently wrong.**
> Predicting $\hat{y} = 0.99$ when the true label is $y = 0$ costs *a lot*.
> Predicting $\hat{y} = 0.51$ when the true label is $y = 0$ costs only *a
> little*. Predicting $\hat{y} = 0.99$ when the true label *is* $y = 1$ — the
> happy case — costs *almost nothing*.

So the model has every reason to push its predicted probability up for the true
class and down for the wrong class.

#### The formula, piece by piece

For a single example $(x_i, y_i)$ with predicted probability $\hat{y}_i$ for
the positive class, cross-entropy is:

$$\ell_i = -\Big[\, y_i \log \hat{y}_i + (1 - y_i)\log(1 - \hat{y}_i)\,\Big]$$

Read it like this:

- **If the true label is $y_i = 1$:** only the first term survives, so
  $\ell_i = -\log \hat{y}_i$. The penalty is the negative log of how much
  probability the model gave to the *correct* class.
- **If the true label is $y_i = 0$:** only the second term survives, so
  $\ell_i = -\log(1 - \hat{y}_i)$. Same idea — the negative log of the
  probability the model gave to *being a 0*.

The total loss is just the average over the dataset:

$$L = \frac{1}{N}\sum_{i=1}^{N} \ell_i
   = -\frac{1}{N}\sum_{i=1}^{N} \Big[\, y_i \log \hat{y}_i + (1 - y_i)\log(1 - \hat{y}_i)\,\Big]$$

#### Why the negative log?

Because $\log$ is the function whose value goes to $-\infty$ as its argument
approaches $0$. So $-\log p$ goes to $+\infty$ as the predicted probability for
the true class approaches zero — *exactly* the "confidently wrong" punishment we
wanted. Conversely, $-\log 1 = 0$: a perfect, fully-confident prediction costs
nothing.

> **Key Insight: lower loss ⇔ better predictions.** That is the entire reason
> we have a loss function — it converts "is the model good?" into a single
> number we can shrink with calculus.


### `[QUICK]` See cross-entropy with your own eyes

Six predictions, three true-label-1 cases and three true-label-0 cases. Watch
the loss explode when the model is *confidently wrong* and shrink to almost
zero when it's *confidently right*.


In [ ]:
import pandas as pd

eps = 1e-12  # numerical safety: log(0) is undefined

cases = [
    # (true label y, predicted P(class=1) y_hat, plain-English description)
    (1, 0.99, "true=1, model very confident it's a 1   (great)"),
    (1, 0.60, "true=1, model leans 1                    (ok)"),
    (1, 0.05, "true=1, model very confident it's a 0   (terrible)"),
    (0, 0.01, "true=0, model very confident it's a 0   (great)"),
    (0, 0.40, "true=0, model leans 0                    (ok)"),
    (0, 0.95, "true=0, model very confident it's a 1   (terrible)"),
]

rows = []
for y, y_hat, note in cases:
    # Cross-entropy for one example.
    loss = -(y * np.log(y_hat + eps) + (1 - y) * np.log(1 - y_hat + eps))
    rows.append({"y": y, "y_hat": y_hat, "loss": round(loss, 3), "case": note})

pd.DataFrame(rows)


### Gradient descent: walking downhill in weight-space

We have a loss $L$. We want it small. **Gradient descent** is the simplest
recipe: keep nudging the weights in the direction that lowers the loss the
fastest.

The update rule is just one line:

$$w \;\leftarrow\; w \;-\; \eta \, \nabla L(w)$$

Let's unpack each symbol so it stops looking intimidating.

#### What is the gradient $\nabla L(w)$?

In one dimension (a single weight $w$), the "gradient" is just the *derivative*
$\dfrac{dL}{dw}$ — the **slope** of the loss curve at the current $w$. If the
slope is positive, the loss goes *up* if you increase $w$, so you should
*decrease* $w$ to go down. If the slope is negative, the loss goes *down* if
you increase $w$, so you should increase $w$. The minus sign in the update rule
encodes exactly that: *step in the direction opposite to the slope*.

In many dimensions (a real network has thousands of weights), $\nabla L$ is a
*vector* — one slope per weight — that points in the direction of **steepest
ascent**. Subtracting it (with a minus sign) means moving in the direction of
**steepest descent**: the fastest possible decrease in $L$ for a small step.

> **One-line summary.** *The gradient tells you which way is uphill;
> gradient descent walks the other way.*

#### What is the learning rate $\eta$?

$\eta$ is just the **step size** — how far you move in the downhill direction
on each update. Too small and you crawl; too large and you overshoot the valley
and bounce around (or even diverge).

| $\eta$ too small | $\eta$ just right | $\eta$ too large |
|---|---|---|
| training takes forever | smooth, steady descent | loss bounces or explodes |
| `loss_curve_` barely moves | classic "elbow" shape | loss zig-zags upward |

> **Mental picture.** A ball rolling on a hilly landscape, where altitude is
> the loss. The gradient is the local steepest-descent direction. $\eta$ is
> the *size of each step* the ball takes downhill before re-checking the
> slope. Adam (the default sklearn solver) is a fancy ball that adjusts its
> step size automatically.

The plot below takes 12 gradient-descent steps on a wiggly 1-D loss. Watch
each red dot land lower than the previous one.


In [ ]:
# Visualise gradient descent on a 1-D non-convex loss surface.
def loss_fn(w):
    return (w - 2.0) ** 2 + 0.5 * np.sin(4 * w)

def loss_grad(w):
    return 2 * (w - 2.0) + 2.0 * np.cos(4 * w)

# Take a few gradient-descent steps starting from w0 = -0.5.
eta = 0.08
w = -0.5
trajectory = [w]
for _ in range(12):
    w = w - eta * loss_grad(w)
    trajectory.append(w)

w_grid = np.linspace(-1.5, 5.0, 400)
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(w_grid, loss_fn(w_grid), color='steelblue', linewidth=2)
ax.scatter(trajectory, [loss_fn(w) for w in trajectory],
           color='indianred', s=70, zorder=5, edgecolor='white')
for i, w in enumerate(trajectory):
    if i % 2 == 0:
        ax.annotate(f'step {i}', (w, loss_fn(w)),
                    textcoords='offset points', xytext=(6, 8), fontsize=9)
ax.set_xlabel('weight  $w$', fontsize=12)
ax.set_ylabel('loss  $L(w)$', fontsize=12)
ax.set_title('Gradient descent on a non-convex 1-D loss', fontsize=13)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Three things to notice — the first two are visible in the plot, the third is
the most common training failure mode.

1. **The trajectory marches downhill.** Each red dot has a smaller $L$ than the
   previous one. That is gradient descent, in pictures.
2. **The surface has multiple local minima.** Different starting weights would
   roll into different valleys. Real neural-network loss surfaces are wildly
   bumpier than this 1-D toy. This is why `MLPClassifier` has a `random_state`
   parameter and why two runs with different seeds give slightly different
   results — the optimiser literally lands somewhere else.
3. **Step size matters.** With a sensible $\eta = 0.08$ the descent is smooth.
   Crank $\eta$ up too high and the trajectory would *overshoot* the valley
   and bounce; turn it down too low and the ball would barely move from where
   it started. Sklearn's `learning_rate_init` is exactly this knob.

The cell below sweeps three learning rates so you can *see* the goldilocks
problem before you ever debug it on a real dataset.


In [ ]:
# Same 1-D loss as before; sweep three learning rates and overlay trajectories.
def gd_trajectory(eta, w0=-0.5, n_steps=15):
    w = w0
    traj = [w]
    for _ in range(n_steps):
        w = w - eta * loss_grad(w)
        traj.append(w)
    return np.array(traj)

w_grid = np.linspace(-1.5, 5.0, 400)
lr_settings = [
    (0.01, 'too small', 'gray'),
    (0.08, 'just right', 'steelblue'),
    (0.45, 'too large',  'indianred'),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, (eta, label, colour) in zip(axes, lr_settings):
    traj = gd_trajectory(eta)
    ax.plot(w_grid, loss_fn(w_grid), color='steelblue', linewidth=1.5, alpha=0.5)
    ax.plot(traj, [loss_fn(w) for w in traj], '-o', color=colour,
            markeredgecolor='white', markersize=7, linewidth=1.2)
    ax.set_title(f'η = {eta}  ({label})', fontsize=12)
    ax.set_xlabel('weight  $w$')
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel('loss  $L(w)$')
plt.suptitle('Same loss surface, three learning rates — pick your poison', fontsize=13)
plt.tight_layout()
plt.show()


### Backpropagation: the chain rule as blame propagation

We now know *what direction to step in* (downhill, opposite the gradient).
But computing $\nabla L$ for a network with thousands of weights used to be
the hard part. The breakthrough — **backpropagation** (Rumelhart, Hinton &
Williams, 1986) — is *a clever, efficient application of the chain rule from
calculus*. Without writing the equations out:

> **Each weight gets blamed proportionally to how much it contributed to the
> error, and the blame flows backward through the network from the output
> toward the inputs.**

#### Walking through it on the 2 → 2 → 1 network from §3

Recall the toy network we did the forward pass on earlier (cell with weights
`W1, W2`). For input `x = [0.5, -0.2]` we got:

- hidden activations $\mathbf{h} = [0.31, 0.17]$
- output $\hat{y} = 0.575$

Now suppose the true label was $y = 1$. The model is *not* terribly wrong (it
already leans 1), but it could lean harder. Cross-entropy is roughly
$-\log(0.575) \approx 0.553$. Where did that loss come from? Backprop
answers in three sweeps:

**1. Blame the output layer.** The error signal at the output is just
$\hat{y} - y = 0.575 - 1 = -0.425$. *Negative* means: the prediction is too
low; raising $\hat{y}$ would reduce the loss. So the gradient of $L$ with
respect to each output weight $W^{(2)}_j$ is

$$\frac{\partial L}{\partial W^{(2)}_j} \;=\; (\hat{y} - y)\, h_j$$

For $W^{(2)}_1$ (the weight on hidden neuron 1, with $h_1 = 0.31$):
$\;-0.425 \times 0.31 \approx -0.132$. We *subtract* $\eta$ times that
from the weight, so $W^{(2)}_1$ goes *up* (because the gradient was negative).
Translation: *"the first hidden neuron was firing in the right direction;
reward it by listening to it more."*

**2. Push the blame back to the hidden layer.** Each hidden neuron's share of
the blame is proportional to how strongly it talks to the output:

$$\text{blame on } h_j \;=\; (\hat{y} - y)\, W^{(2)}_j$$

That blame is then *gated by the activation*: ReLU only passed through the
neuron if $z > 0$, so a silent neuron gets zero blame (and zero update). Live
neurons get scaled blame.

**3. Blame the input weights.** With each hidden neuron's share of the blame
in hand, the same rule fires again: each input weight's gradient is
*(blame on the neuron it feeds) × (its input value)*. The chain rule has now
walked all the way from the output back to the original features, in one
single backward pass.

#### Why is this efficient?

A naive computation would re-derive every gradient from scratch, costing time
proportional to *(number of weights)²*. Backprop reuses the activations stored
during the forward pass and shares the blame down the layers, costing time
proportional to *(number of weights)* — the same as one forward pass. That
linear scaling is what made deep learning practical.

> **The library does the calculus for you.** `MLPClassifier` runs a forward
> pass, computes the loss, runs a backward pass, updates the weights with the
> gradients it just computed, and repeats. **You** control the architecture,
> the learning rate, and when to stop. The chain rule is automatic.

> **Key Insight: backprop = chain rule + bookkeeping.** Forward pass produces
> predictions. Backward pass distributes blame. Update step uses that blame to
> nudge every weight a tiny bit toward a better prediction. Run that loop
> a few hundred times and a randomly-initialised network learns Nepal damage
> grades, MNIST digits, or — at the right scale — language.


### Watching the loss descend — for real

Let's fit a small MLP on moons and plot its `loss_curve_` (`MLPClassifier` exposes
the training loss at every iteration).

In [ ]:
m_moons = MLPClassifier(hidden_layer_sizes=(16,), activation='relu',
                        max_iter=400, random_state=49,
                        solver='adam').fit(X_moons, y_moons)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(m_moons.loss_curve_, color='steelblue', linewidth=2)
ax.set_xlabel('training iteration', fontsize=12)
ax.set_ylabel('cross-entropy loss', fontsize=12)
ax.set_title(f'Loss curve of MLPClassifier((16,)) on moons '
             f'— final accuracy {m_moons.score(X_moons, y_moons):.3f}', fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### `[DISCUSS]` The same architecture can land in different valleys

Train the *same* MLP twice with two different `random_state`s and watch the final
loss differ a little. This is non-convex optimisation in action — and a reason to
report results across multiple seeds when you submit a final-project model.

In [ ]:
for seed in [1, 49]:
    m = MLPClassifier(hidden_layer_sizes=(16,), activation='relu',
                      max_iter=400, random_state=seed).fit(X_moons, y_moons)
    print(f"seed={seed:>3}  final loss = {m.loss_curve_[-1]:.4f}   accuracy = {m.score(X_moons, y_moons):.3f}")

### Engineering knobs

In practice, training is configured by a handful of parameters:

- **Mini-batch SGD** — instead of computing $\nabla L$ on the whole dataset,
  estimate it from a small *batch* (controlled by `batch_size`). This is what
  makes training fast on large data.
- **Adam** (`solver='adam'`) — a smarter version of SGD that adapts the learning
  rate per weight. Good default for tabular MLPs.
- **`early_stopping=True`** — sklearn holds out a small validation slice and stops
  training when validation loss stops improving. Prevents overfitting by training
  budget rather than by careful $L_2$ tuning. Recommended.

> **Key Insight: Backprop = chain rule = blame propagation. The library does the
> calculus; you control architecture, learning rate, and stopping.**

## 5. Application: MLP vs Random Forest on Nepal Earthquake Damage

Time for the honest comparison. We pick up exactly where Week 9 left off — the same
50,000 buildings, the same `damage_grade ∈ {1, 2, 3}` target, the same train/test
split. We will:

1. Build an `MLPClassifier` pipeline with proper preprocessing.
2. Demonstrate the **scaling footgun** (the #1 newbie mistake).
3. Train a final MLP and report its metrics.
4. Refit the Week-9 Random Forest on the *same* split and compare side-by-side.
5. Interpret the result honestly — including when MLP would have won.

> **Note.** We load the CSV directly from Week 9's data folder so the comparison is
> byte-for-byte fair; do not duplicate the file. Refer to
> `Week09_.../data/README.md` for dataset attribution and schema.

In [ ]:
# Load directly from Week 9's data folder.
DATA = '../Week09_Decision_Trees_and_Random_Forests/data/nepal_buildings_sample.csv'
df = pd.read_csv(DATA)

# Same target and split as Week 9.
y = df['damage_grade']
X = df.drop(columns=['building_id', 'damage_grade'])

cat_cols = X.select_dtypes(include='object').columns.tolist()
num_cols = X.select_dtypes(exclude='object').columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=49
)
print(f"train: {X_train.shape}, test: {X_test.shape}")
print(f"categorical columns: {len(cat_cols)},  numeric columns: {len(num_cols)}")
print(f"class balance (train): {y_train.value_counts(normalize=True).round(3).to_dict()}")

### Building the pipeline

Two preprocessing pieces matter for an MLP on tabular data:

1. **Encoding categoricals.** We use `OrdinalEncoder` to match Week 9's setup
   exactly. *Strictly speaking*, one-hot encoding is the safer default for an MLP
   (it doesn't impose a fake ordering on unordered categories), but ordinal keeps
   the apples-to-apples comparison and the MLP can still learn from the codes.
   If you want to squeeze a couple extra points later, swap `OrdinalEncoder`
   for `OneHotEncoder(handle_unknown='ignore')`.
2. **Scaling.** Unlike trees, **MLPs require scaled inputs**. We apply
   `StandardScaler` to *all* features (numeric pass-throughs and ordinal-encoded
   categoricals) after the encoder. We will see why this matters in a moment.

Everything sits inside one `Pipeline` so there is no chance of leakage between
fit and transform.

In [ ]:
def build_mlp_pipeline(*, scale, hidden=(64, 32), max_iter=200,
                       early_stopping=False, random_state=49):
    pre = ColumnTransformer(
        transformers=[
            ('cat', OrdinalEncoder(handle_unknown='use_encoded_value',
                                   unknown_value=-1), cat_cols),
        ],
        remainder='passthrough',
    )
    steps = [('pre', pre)]
    if scale:
        steps.append(('scaler', StandardScaler()))
    steps.append(('mlp', MLPClassifier(hidden_layer_sizes=hidden,
                                       activation='relu',
                                       solver='adam',
                                       max_iter=max_iter,
                                       early_stopping=early_stopping,
                                       random_state=random_state)))
    return Pipeline(steps)

### `[TOGETHER]` The scaling experiment

We fit the *same* MLP twice — once without `StandardScaler`, once with — and
overlay the two `loss_curve_`s. Watch what happens.

In [ ]:
mlp_unscaled = build_mlp_pipeline(scale=False, hidden=(64, 32), max_iter=200).fit(X_train, y_train)
mlp_scaled   = build_mlp_pipeline(scale=True,  hidden=(64, 32), max_iter=200).fit(X_train, y_train)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(mlp_unscaled.named_steps['mlp'].loss_curve_,
        color='indianred', linewidth=2, label='unscaled inputs')
ax.plot(mlp_scaled.named_steps['mlp'].loss_curve_,
        color='steelblue', linewidth=2, label='StandardScaler')
ax.set_xlabel('training iteration', fontsize=12)
ax.set_ylabel('cross-entropy loss', fontsize=12)
ax.set_title('Same MLP, same data, same seed — scaling is the difference', fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"unscaled  test accuracy = {mlp_unscaled.score(X_test, y_test):.4f}")
print(f"scaled    test accuracy = {mlp_scaled.score(X_test, y_test):.4f}")

> **Key Insight: MLPs assume the input features are roughly mean-0, unit-variance.**
> Forget the scaler and you forget half the model. This is the single most common
> beginner mistake with neural networks; the trees of Week 9 made you forget that
> scaling exists.

### The hyperparameters that matter

| Knob | What it controls | Sensible default |
|---|---|---|
| `hidden_layer_sizes` | architecture (width, depth) | start `(64, 32)` for tabular |
| `activation` | hidden-layer non-linearity | `'relu'` |
| `solver` | optimiser | `'adam'` (`'lbfgs'` for tiny data) |
| `alpha` | $L_2$ regularisation strength | `1e-4` (Week-8 `C`, Week-9 pruning — same dial) |
| `learning_rate_init` | step size $\eta$ | leave default; only tune if loss misbehaves |
| `early_stopping` | hold-out + auto-stop | `True` whenever you have enough data |
| `batch_size` | mini-batch size | `'auto'` |
| `random_state` | seed | fix it, always |

In [ ]:
# Final scaled MLP with early stopping, fit on the full training data.
final_mlp = build_mlp_pipeline(scale=True, hidden=(64, 32),
                               max_iter=300, early_stopping=True,
                               random_state=49).fit(X_train, y_train)

y_pred_mlp = final_mlp.predict(X_test)
print(f"MLP test accuracy: {accuracy_score(y_test, y_pred_mlp):.4f}\n")
print(classification_report(y_test, y_pred_mlp, digits=3))

In [ ]:
# Row-normalised confusion matrix — same widget Week 9 used.
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_estimator(final_mlp, X_test, y_test,
                                      normalize='true', cmap='Blues',
                                      ax=ax, colorbar=False)
ax.set_title('MLP — row-normalised confusion matrix', fontsize=12)
plt.tight_layout()
plt.show()

### Refitting Week 9's Random Forest on the same split

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=49, n_jobs=-1)
rf_pre = ColumnTransformer(
    [('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols)],
    remainder='passthrough',
)
rf_pipe = Pipeline([('pre', rf_pre), ('rf', rf)]).fit(X_train, y_train)

y_pred_rf = rf_pipe.predict(X_test)
print(f"Random Forest test accuracy: {accuracy_score(y_test, y_pred_rf):.4f}\n")
print(classification_report(y_test, y_pred_rf, digits=3))

In [ ]:
import time

def time_fit(make_pipe):
    t0 = time.time()
    p = make_pipe().fit(X_train, y_train)
    return p, time.time() - t0

mlp_p, t_mlp = time_fit(lambda: build_mlp_pipeline(scale=True, hidden=(64, 32),
                                                  max_iter=300, early_stopping=True,
                                                  random_state=49))
rf_p,  t_rf  = time_fit(lambda: Pipeline([
    ('pre', ColumnTransformer(
        [('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols)],
        remainder='passthrough')),
    ('rf', RandomForestClassifier(n_estimators=200, random_state=49, n_jobs=-1)),
]))

def per_class_recall(y_true, y_pred, cls):
    mask = y_true == cls
    return (y_pred[mask] == cls).mean()

rows = []
for name, model, t in [('Random Forest', rf_p, t_rf),
                       ('MLP (64, 32)',  mlp_p, t_mlp)]:
    yp = model.predict(X_test)
    rows.append({
        'model':         name,
        'train_acc':     round(model.score(X_train, y_train), 4),
        'test_acc':      round(accuracy_score(y_test, yp), 4),
        'recall_grade1': round(per_class_recall(y_test.values, yp, 1), 3),
        'recall_grade3': round(per_class_recall(y_test.values, yp, 3), 3),
        'fit_time_s':    round(t, 2),
    })

comparison = pd.DataFrame(rows).set_index('model')
comparison

### The honest interpretation

> **Key Insight: Random Forest matching or beating MLP on this task is a real,
> reproducible result — not a teaching artifact.** Three reasons:
>
> 1. **The data is tabular and heterogeneous.** Categorical materials, ordinal
>    location codes, small-integer counts. Trees handle this natively; MLPs only
>    see a vector of scaled numbers.
> 2. **The dataset is medium-sized** (~50k rows, 38 features). Tree ensembles
>    dominate this regime. Neural networks need orders of magnitude more data —
>    or a different data shape — to pull ahead.
> 3. **The signal lives in simple feature interactions** (location × material × age).
>    That is exactly what trees specialise in.

> **Example: Civil Engineering — when MLPs (and deeper variants) win.**
> - **Drone imagery of post-quake buildings** → CNNs (convolutions exploit spatial
>   structure that no Random Forest can match).
> - **Accelerometer time series from structural-health-monitoring sensors** →
>   1-D CNNs / RNNs / Transformers.
> - **Inspector free-text reports** → Transformers (BERT-style language models).
> - **Million-row tabular datasets with rich interactions** — modern tabular deep
>   models can beat trees, but the gap is small and the engineering cost is large.

## 6. Regression, Comparison, Takeaways & Practice

### MLP for regression — the same network, a different output

Switching from classification to regression changes only two things:

- The output activation goes from sigmoid (or softmax) to **linear** (identity).
- The loss goes from cross-entropy to **mean-squared error** (MSE).

Everything else — the hidden layers, ReLU, training, scaling — is identical.

In [ ]:
# Synthetic regression problem (mirrors Week 9 §6's make_regression aside).
X_reg, y_reg = make_regression(n_samples=400, n_features=1, noise=15.0, random_state=49)

mlp_reg = Pipeline([
    ('scale', StandardScaler()),
    ('mlp',   MLPRegressor(hidden_layer_sizes=(32, 16), activation='relu',
                           max_iter=2000, random_state=49)),
]).fit(X_reg, y_reg)

x_grid = np.linspace(X_reg.min(), X_reg.max(), 200).reshape(-1, 1)
y_grid = mlp_reg.predict(x_grid)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(X_reg, y_reg, color='steelblue', edgecolor='white',
           s=35, alpha=0.7, label='data')
ax.plot(x_grid, y_grid, color='indianred', linewidth=2.5, label='MLPRegressor')
ax.set_xlabel('$x$', fontsize=12); ax.set_ylabel('$y$', fontsize=12)
ax.set_title(f'MLPRegressor on synthetic data — R² = {mlp_reg.score(X_reg, y_reg):.3f}',
             fontsize=12)
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### The updated model zoo

Extending Week 9's comparison table to include MLPs:

| Property | Naive Bayes | SVM (RBF) | Random Forest | **MLP** |
|---|---|---|---|---|
| Decision style | Probabilistic | Geometric (margin) | Sequential rules | **Layered transforms** |
| Needs scaling? | No | Yes | No | **Yes** |
| Handles mixed types? | Limited | One-hot | Native | **Encode + scale** |
| Captures non-linearity? | Limited | With kernel | For free | **For free** |
| Interpretable? | Moderate | Hard | Moderate (importances) | **Hard** |
| Best on tabular CE data? | Sometimes | Sometimes | **Yes** | Sometimes |
| Best on images / sequences? | No | No | No | **Yes (with CNN/RNN/Transformer variants)** |
| Tuning effort | Low | Medium | Low | **High** |

### When to use what — a practical decision flowchart

- **Tabular data, mixed types, < ~1M rows, need interpretability** → **Random Forest**. Always start here.
- **Tabular data, you've squeezed RF and want a few more points** → gradient boosting (XGBoost / LightGBM), then maybe MLP.
- **Linear-ish problem, need calibrated probabilities** → Logistic regression.
- **Clean numeric data, low-D, sharp curved boundary** → SVM with RBF.
- **Images** → CNNs (Keras/PyTorch).
- **Sequences (text, time series)** → RNN / Transformer (Keras/PyTorch).
- **Tiny tabular dataset (< ~100 rows)** → Don't use MLP. Use logistic regression or a single tree.

### The graduation map: when sklearn is no longer enough

| Data shape | Tool | What it adds |
|---|---|---|
| **Images** (drone photos of damaged buildings) | **CNN** in Keras/PyTorch | Convolutional layers learn spatial filters automatically. |
| **Time series** (accelerometer SHM streams) | **1-D CNN, LSTM, or Transformer** | Layers that exploit temporal ordering. |
| **Text** (inspector reports, codes) | **Transformer** (BERT, GPT-family) | Attention mechanisms over sequences. |
| **Multi-modal** (images + tabular + text together) | **Custom architecture** in PyTorch | Combine specialised encoders for each modality. |

The library leap (Keras / PyTorch) is mostly **engineering**: GPU support, custom
layer types, training-loop control, dataset pipelines. The **concepts** — forward
pass, loss, gradient descent, backprop — are *exactly* what you learned today. You
already know neural networks; deeper networks just have more layers and more
specialised layer types.

### Key Takeaways

1. A **neuron** is a weighted sum + bias + activation. A **network** is many neurons stacked in layers.
2. **Hidden layers are *learned* feature transforms** — the deep-learning analog of SVM kernels.
3. Without a non-linear activation, stacking neurons just gives one big linear model.
4. **Training** = minimise a loss function via gradient descent; **backprop** is the chain rule that distributes blame to weights.
5. **MLPs require scaled inputs.** This is the single most common newbie mistake.
6. The `MLPClassifier` knobs that actually matter: `hidden_layer_sizes`, `alpha`, `early_stopping`, `random_state`.
7. On medium-sized **tabular** CE data, **Random Forest typically matches or beats MLP** — and gives you feature importances for free.
8. **MLPs shine when data is *not* tabular**: images (CNN), sequences (RNN/Transformer), or millions of rows.
9. The concepts you learned today are the *same* ones that power deep learning at scale. Graduating to Keras / PyTorch is engineering, not new theory.
10. Always have a **non-neural baseline** (RF, gradient boosting, logistic regression) before claiming a neural network "works".

### A practical workflow

1. **Start with the Week-9 Random Forest baseline.** Always.
2. Reach for an MLP only if you have a reason — large data, a non-tabular component, or RF has plateaued.
3. Build a `Pipeline`: encoder (`OrdinalEncoder` or `OneHotEncoder`) → `StandardScaler` → `MLPClassifier`.
4. Start small: `(64, 32)` ReLU, `early_stopping=True`, default `alpha`, fixed `random_state`.
5. **Plot `loss_curve_`.** If it doesn't descend cleanly, your inputs aren't scaled or your learning rate is wrong.
6. **Compare against your RF on the same split.** If MLP doesn't clearly win, ship the RF.
7. Only if the gap matters: tune architecture and `alpha`, or graduate to Keras/PyTorch.

> **Practical Recommendation: Try Random Forest first. Reach for MLP — and deep
> learning — only when the data shape or scale demands it.**

## [PRACTICE] Practice Exercises

Try these on your own. Exercises 4 and 5 are excellent warm-ups for your final project.

1. **Exercise 1 (Warm-up — activations & forward pass).** Define a 2 → 3 → 1 network
   *by hand* (ReLU hidden, sigmoid output) with weights of your choosing (small
   numbers between $-1$ and $1$). Compute the forward pass for inputs $[1.0, -1.0]$
   and $[0.0, 0.5]$. In a markdown cell, explain in 2–3 sentences what each hidden
   neuron is detecting based on the sign and magnitude of its weights.

2. **Exercise 2 (The scaling footgun).** Take `make_moons(n_samples=400, noise=0.20)`
   and **multiply one feature by 1000** (simulate having one feature in millimetres
   and another in years). Fit `MLPClassifier((16,), max_iter=2000, random_state=49)`
   twice — once without scaling, once with `StandardScaler` in a `Pipeline`. Plot
   both decision boundaries side-by-side. In a paragraph, explain why scaling
   matters for MLPs but not for Random Forests.

3. **Exercise 3 (Tuning regularisation).** On the Nepal pipeline, build a
   `validation_curve` sweeping `alpha ∈ [1e-5, 1e-4, 1e-3, 1e-2, 1e-1]` for an
   `MLPClassifier((64, 32))`. Plot train and CV accuracy. Identify the sweet spot.
   **Bonus:** repeat with `hidden_layer_sizes ∈ [(8,), (32,), (64, 32), (128, 64, 32)]`.
   Which architecture generalises best, and is the gap to Random Forest closing?

4. **Exercise 4 (Honest comparison).** Train an MLP, a Random Forest, and a Logistic
   Regression on the Nepal dataset using the *same* train/test split. Build a
   single comparison table containing test accuracy, per-class recall, fit time,
   and prediction time. In a paragraph, recommend a model for a Nepali ministry
   that needs to (a) prioritise retrofit budget and (b) defend the recommendation
   in a public hearing. Justify with **interpretability and accuracy together**,
   not either alone.

5. **Exercise 5 (Final-project warm-up).** Pick a tabular dataset relevant to your
   final-project domain. (i) Train a Random Forest baseline. (ii) Train an MLP.
   (iii) Report metrics. (iv) **In one paragraph**, decide which model you would
   actually deploy and *why* — referencing data size, feature types, and
   interpretability needs. If neither model beats the other clearly, say so
   explicitly: *"RF and MLP are within noise; I'll ship RF for interpretability."*
   That sentence is a *correct* engineering decision and worth full credit.

In [ ]:
# Your code here

---

### Questions?

**Dr. Eyuphan Koc**
eyuphan.koc@bogazici.edu.tr